In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
import pickle

from pathlib import Path

folder = "C:/Users/JesseOnu/fpl sql rework/gws"

cols_to_keep = ['player_code', 'gameweek', 
       'web_name', 'position_id',
       'minutes', 'goals_scored', 'assists', 'clean_sheets',
       'yellow_cards', 'red_cards', 'bonus', 'bps', 'total_points', 'xG', 'xA',
       'xGI', 'DefCon',
       'corners_indirect_freekicks_order', 'direct_freekicks_order',
       'penalties_order', 'season']

def load_season_data(path):
    files = os.listdir(path)

    dataframes = []
    for f in files:
        filepath = os.path.join(path, f)
        df = pd.read_csv(filepath, usecols = cols_to_keep)
        dataframes.append(df)

    df_combined = pd.concat(dataframes, ignore_index=True)

    return df_combined.reset_index(drop=True)

df = load_season_data(folder)
df = df[df['season'] ==2526]

stats = df.groupby(
    ['player_code', 'web_name']).agg({
    'minutes': 'sum', 
    'goals_scored':'sum',
    'assists': 'sum', 
    'xA':'sum',
    'xG':'sum',
    'xA':'sum', 
    'xGI':'sum', 
    'DefCon':'sum',
    'position_id':'max'}) 
max_mins = stats['minutes'].max()
stats['min_perc'] = (stats['minutes']/max_mins)
stats = stats[stats['min_perc']>0.29]

stats['xGp90'] = (90* stats['xG'] / stats['minutes']).fillna(0).round(2)
stats['xAp90'] = (90* stats['xA'] / stats['minutes']).fillna(0).round(2)
stats['GoalInv'] =  stats['goals_scored'] + stats['assists'] 
stats['GIp90'] = (90* stats['GoalInv'] / stats['minutes']).fillna(0).round(2)
stats['xGIp90'] = (90* stats['xGI'] / stats['minutes']).fillna(0).round(2)
stats['defconp90'] = (90* stats['DefCon'] / stats['minutes']).fillna(0).round(2)
stats['xgisafe'] = stats['xGI'].apply(lambda x: x if x>0 else 0.01)
stats['xasafe'] = stats['xA'].apply(lambda x: x if x>0 else 0.01)
stats['xgi_performance_pct'] = ((stats['GoalInv'] - stats['xgisafe']) / stats['xgisafe']).round(2)
stats['xa_performance_pct'] = ((stats['assists'] - stats['xasafe']) / stats['xasafe']).round(2)
stats['xgi_performance_pct'] = stats['xgi_performance_pct'].clip(-1.0, 1.0)
stats['xa_performance_pct'] = stats['xa_performance_pct'].clip(-1.0, 1.0)

position_config = {
    4: {"features":['xGp90', 'xAp90', 'xgi_performance_pct', 'xa_performance_pct'], 
        "k": 4},
    3: {"features": ['xGp90', 'xAp90', 'defconp90', 'xgi_performance_pct', 'xa_performance_pct'], 
        "k": 7},
    2: {"features": ['xGp90', 'xAp90', 'defconp90'], 
        "k": 3}
}

all_features = ['xGp90', 'xAp90', 'defconp90', 'xgi_performance_pct', 'xa_performance_pct']
scaler = MinMaxScaler()
stats_normalized = stats.copy()
stats_normalized[all_features] = scaler.fit_transform(stats[all_features])

# Step 3: For each position, run K-means
centroids_dict = {}
for position_id, config in position_config.items():
    position_data = stats_normalized[stats_normalized['position_id'] == position_id]
    features = config['features']
    k = config['k']
    
    # Your K-means code here
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(position_data[features])
    
    centroids_dict[position_id] = kmeans.cluster_centers_
    print(f"Position {position_id}: {kmeans.cluster_centers_.shape}")

training_output = {
    'centroids': centroids_dict,
    'scaler': scaler,
    'position_config': position_config,
    'archetype_names': {
        4: {0: "Goal Threat (FWD)", 1: "Well Rounded (FWD)", 2: "Elite Finisher (FWD)", 3: "Creative Forward (FWD)"},
        3: {0: "DefCon (MID)", 1: "Goal Threat (MID)", 2: "Creative Playmaker (MID)", 3: "Well Balanced (MID)", 4: "Playmaker DefCon Potential (MID)", 5: "DefCon Efficient Output(MID)", 6: "Box to Box( MID)"},
        2: {0: "Attacking Potential (DEF)", 1: "Balanced (DEF)", 2: "DefCon (DEF)"}
    }
}
 
# Save
with open('C:/Users/JesseOnu/fpl sql rework/fpl classifications/fpl_classification_training.pkl', 'wb') as f:
    pickle.dump(training_output, f)

print("Training complete. Saved to fpl_classification_training.pkl")

Position 4: (4, 4)
Position 3: (7, 5)
Position 2: (3, 3)
Training complete. Saved to fpl_classification_training.pkl


C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

In [2]:
print(stats.columns.tolist())

['minutes', 'goals_scored', 'assists', 'xA', 'xG', 'xGI', 'DefCon', 'position_id', 'min_perc', 'xGp90', 'xAp90', 'GoalInv', 'GIp90', 'xGIp90', 'defconp90', 'xgisafe', 'xasafe', 'xgi_performance_pct', 'xa_performance_pct']


In [3]:
with open('C:/Users/JesseOnu/fpl sql rework/fpl classifications/fpl_classification_training.pkl', 'rb') as f:
    training_output = pickle.load(f)
print("Keys:", training_output.keys())
print("Centroids keys:", training_output['centroids'].keys())
print("Position 4 centroids shape:", training_output['centroids'][4].shape)
print("Position 3 centroids shape:", training_output['centroids'][3].shape)
print("Position 2 centroids shape:", training_output['centroids'][2].shape)
print("Archetype names:", training_output['archetype_names'])

Keys: dict_keys(['centroids', 'scaler', 'position_config', 'archetype_names'])
Centroids keys: dict_keys([4, 3, 2])
Position 4 centroids shape: (4, 4)
Position 3 centroids shape: (7, 5)
Position 2 centroids shape: (3, 3)
Archetype names: {4: {0: 'Goal Threat (FWD)', 1: 'Well Rounded (FWD)', 2: 'Elite Finisher (FWD)', 3: 'Creative Forward (FWD)'}, 3: {0: 'DefCon (MID)', 1: 'Goal Threat (MID)', 2: 'Creative Playmaker (MID)', 3: 'Well Balanced (MID)', 4: 'Playmaker DefCon Potential (MID)', 5: 'DefCon Efficient Output(MID)', 6: 'Box to Box( MID)'}, 2: {0: 'Attacking Potential (DEF)', 1: 'Balanced (DEF)', 2: 'DefCon (DEF)'}}


In [4]:
# Forwards
print("=== FORWARDS ===")
forward_centroids = training_output['centroids'][4]
forward_names = training_output['archetype_names'][4]
for cluster_id, name in forward_names.items():
    print(f"Cluster {cluster_id} ({name}): {forward_centroids[cluster_id]}")

# Midfielders
print("\n=== MIDFIELDERS ===")
mid_centroids = training_output['centroids'][3]
mid_names = training_output['archetype_names'][3]
for cluster_id, name in mid_names.items():
    print(f"Cluster {cluster_id} ({name}): {mid_centroids[cluster_id]}")

# Defenders
print("\n=== DEFENDERS ===")
def_centroids = training_output['centroids'][2]
def_names = training_output['archetype_names'][2]
for cluster_id, name in def_names.items():
    print(f"Cluster {cluster_id} ({name}): {def_centroids[cluster_id]}")

=== FORWARDS ===
Cluster 0 (Goal Threat (FWD)): [0.60826211 0.12839506 0.50777778 0.13388889]
Cluster 1 (Well Rounded (FWD)): [0.39209402 0.11666667 0.63041667 0.96791667]
Cluster 2 (Elite Finisher (FWD)): [0.69088319 0.11358025 0.59611111 0.97333333]
Cluster 3 (Creative Forward (FWD)): [0.17948718 0.23703704 0.36666667 0.34166667]

=== MIDFIELDERS ===
Cluster 0 (DefCon (MID)): [0.12145749 0.2        0.74539332 0.30657895 0.15710526]
Cluster 1 (Goal Threat (MID)): [0.30576923 0.21111111 0.48195543 0.71325    0.95225   ]
Cluster 2 (Creative Playmaker (MID)): [0.24804905 0.47342995 0.49101366 0.65652174 0.70934783]
Cluster 3 (Well Balanced (MID)): [0.30936455 0.36811594 0.43706436 0.36934783 0.38130435]
Cluster 4 (Playmaker DefCon Potential (MID)): [0.10030166 0.21045752 0.73633019 0.72058824 0.78882353]
Cluster 5 (DefCon Efficient Output(MID)): [0.09134615 0.14444444 0.75076384 0.9346875  1.        ]
Cluster 6 (Box to Box( MID)): [0.1348528  0.29300412 0.69736667 0.52       0.54222222]


In [5]:
# Check the stats dataframe BEFORE normalization
print("Stats shape:", stats.shape)
print("\nFeature ranges BEFORE normalization:")
all_features = ['xGp90', 'xAp90', 'defconp90', 'xgi_performance_pct', 'xa_performance_pct']
for feat in all_features:
    print(f"{feat}: min={stats[feat].min()}, max={stats[feat].max()}, mean={stats[feat].mean()}")

print("\nAny inf or extreme values?")
print(stats[all_features].describe())

Stats shape: (322, 19)

Feature ranges BEFORE normalization:
xGp90: min=0.0, max=0.78, mean=0.13046583850931678
xAp90: min=0.0, max=0.45, mean=0.08552795031055901
defconp90: min=0.0, max=13.91, mean=7.125590062111802
xgi_performance_pct: min=-1.0, max=1.0, mean=0.0322360248447205
xa_performance_pct: min=-1.0, max=1.0, mean=0.12757763975155278

Any inf or extreme values?
            xGp90       xAp90   defconp90  xgi_performance_pct  \
count  322.000000  322.000000  322.000000           322.000000   
mean     0.130466    0.085528    7.125590             0.032236   
std      0.136861    0.069398    3.061588             0.578121   
min      0.000000    0.000000    0.000000            -1.000000   
25%      0.040000    0.030000    5.432500            -0.310000   
50%      0.085000    0.070000    7.310000             0.060000   
75%      0.170000    0.130000    9.240000             0.457500   
max      0.780000    0.450000   13.910000             1.000000   

       xa_performance_pct  
coun

In [6]:
stats = stats.reset_index()
for position_id in [4, 3, 2]:
    position_name = {4: "FORWARDS", 3: "MIDFIELDERS", 2: "DEFENDERS"}[position_id]
    print(f"\n=== {position_name} ===\n")
    
    position_stats = stats[stats['position_id'] == position_id].copy()
    position_normalized = stats_normalized[stats_normalized['position_id'] == position_id]
    
    features = position_config[position_id]['features']
    k = position_config[position_id]['k']
    
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_assignments = kmeans.fit_predict(position_normalized[features])
    
    position_stats['cluster'] = cluster_assignments
    
    # Show players per cluster
    for cluster_id in range(k):
        cluster_players = position_stats[position_stats['cluster'] == cluster_id]
        archetype = training_output['archetype_names'][position_id][cluster_id]
        print(f"Cluster {cluster_id} ({archetype}) - {len(cluster_players)} players")
        print(cluster_players[['web_name', 'xGp90', 'xAp90', 'defconp90', 'xgi_performance_pct', 'xa_performance_pct']].head(10))
        print()


=== FORWARDS ===

Cluster 0 (Goal Threat (FWD)) - 9 players
      web_name  xGp90  xAp90  defconp90  xgi_performance_pct  \
1      Welbeck   0.49   0.06       4.48                 0.02   
49     Solanke   0.27   0.03       6.41                -0.09   
63      Nmecha   0.61   0.13       4.82                -0.21   
136   Gyökeres   0.50   0.08       3.45                 0.06   
148     Mateta   0.59   0.05       5.09                -0.24   
183   Flemming   0.41   0.02       5.05                 0.32   
278     Thiago   0.56   0.05       5.87                 0.03   
309  Kroupi.Jr   0.45   0.08       6.06                 0.32   
315      Barry   0.39   0.02       3.51                -0.07   

     xa_performance_pct  
1                 -0.29  
49                -1.00  
63                -0.37  
136               -0.48  
148               -1.00  
183               -1.00  
278               -0.45  
309               -1.00  
315               -1.00  

Cluster 1 (Well Rounded (FWD)) - 12 p

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
